# Flash Flash Revolution Contested Difficulties - Silver Layer

This notebook transforms contested difficulty change records from bronze into analytics-ready silver tables with reliability scores.

## Pipeline Overview

**Bronze → Silver Transformation:**
1. Clean and parse difficulty change records
2. Calculate reliability weights for each assessment
3. Apply data quality filters (valid difficulty range 1-116)
4. Aggregate reliability by difficulty level
5. Apply ordinal smoothing to respect sequential nature of difficulties

**Output Tables:**
- `acubed.ffr.silver__contested-difficulties` - Historical change records with reliability weights
- `acubed.ffr.silver__reliability-scores` - Aggregated reliability by difficulty level (1-120)

---

## Reliability Weight Calculation

Each contested assessment receives a reliability weight based on:

**Components:**
1. **Stability Score**: 1/(num_changes+1) - fewer changes = more stable
2. **Recency Score**: Dynamic decay from 1.0 (current year) to 0.6 (4+ years ago)
3. **Acceptance Score**: 1.0 if matches playlist, 0.5 if differs
4. **Magnitude Confidence**: 1.0 for ±1-2, 0.8 for ±3-5, 0.6 for >±5
5. **Resolution Bonus**: +0.2 if marked "(resolved)"

**Formula:**
```
reliability_weight = (stability × recency × acceptance × magnitude) + resolution_bonus
```

---

## Ordinal Smoothing

Difficulty is an **ordinal variable** - nearby difficulties should have similar reliability scores.

**Method:**
- 5-point rolling window average (±2 difficulties)
- Smooths out noise while preserving trends
- Interpolates meaningful values for uncontested difficulties
- Default reliability for missing difficulties: 0.20

**Effect:**
- Raw range: 0.12 - 0.40
- Smoothed range: 0.19 - 0.33
- Uncontested difficulties inherit from neighbors

---

## Data Quality Filters

Records excluded from silver layer:
- Null `new_difficulty` (from "change" notation)
- `new_difficulty = 0` (unrealistic minimum)
- `new_difficulty > 120` (unrealistic maximum)

**Valid range:** 1-116 (actual difficulty spread in FFR)

---

## Reliability Scores Table Schema

**Table:** `acubed.ffr.silver__reliability-scores`

**Columns:**
- `new_difficulty` (int): Difficulty level 1-120 (complete range)
- `avg_reliability` (double): Smoothed average reliability [0.19 - 0.33]
- `stddev_reliability` (double): Variability in assessments (null if uncontested)
- `num_assessments` (long): Number of contested assessments (0 = uncontested)
- `total_reliability` (double): Sum of reliability weights
- `avg_recency` (double): Average recency score [0.6 - 1.0]
- `avg_acceptance` (double): Average acceptance rate [0.5 - 1.0]

**Coverage:** All 120 difficulty levels guaranteed (gaps filled with defaults)

---

## Usage

The reliability scores join to playlist or features via difficulty level:

```python
df_features.join(
    spark.table('acubed.ffr.silver__reliability-scores'),
    df_features.difficulty == col('new_difficulty'),
    'left'
)
```

Use `avg_reliability` as **sample weights** during model training:
- Higher values = more confident in difficulty rating
- Lower values = less confident (reduce training influence)

---

## Source Tables

**Bronze:**
- `acubed.ffr.bronze__contested-difficulties` - Raw change records from Google Sheets

**Silver (dependencies):**
- `acubed.ffr.silver__playlist` - Current difficulty ratings for acceptance score

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    regexp_replace,
    to_date,
    regexp_extract,
    trim,
    lit,
    current_timestamp,
    substring,
    expr
)
from delta.tables import DeltaTable

In [0]:
# Load bronze table (note: table name has hyphens, requires backticks)
df_bronze = spark.table("acubed.ffr.`bronze__contested-difficulties`")

print("="*60)
print("BRONZE DATA SUMMARY")
print("="*60)
print(f"Total rows: {df_bronze.count()}")
print(f"\nSchema:")
df_bronze.printSchema()

# Show sample data
print("\nSample data (first 10 rows):")
display(df_bronze.limit(10))

In [0]:
# Clean and transform the data
from pyspark.sql.window import Window
from pyspark.sql.functions import last, count as count_func, row_number, year as year_func

df_transformed = df_bronze

# 1. Clean date field - remove 'v' prefixes/suffixes and convert to date
# Use SQL try_to_date via expr to handle invalid dates gracefully (returns null for unparseable values)
df_transformed = df_transformed.withColumn(
    "date_cleaned",
    trim(regexp_replace(col("date"), "v", ""))  # Remove all 'v' characters and trim
)

df_transformed = df_transformed.withColumn(
    "date_parsed",
    expr("try_to_date(date_cleaned, 'yyyy/MM/dd')")  # Use SQL try_to_date via expr
).drop("date_cleaned")

# 1b. Fill missing dates positionally using forward fill within each source_sheet
# Add a row number to preserve original order
window_spec = Window.partitionBy("source_sheet").orderBy(lit(1))
df_transformed = df_transformed.withColumn("row_num", expr("row_number() OVER (PARTITION BY source_sheet ORDER BY (SELECT NULL))"))

# Use last() function to forward fill dates (carry forward the last non-null date)
date_fill_window = Window.partitionBy("source_sheet").orderBy("row_num").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_transformed = df_transformed.withColumn(
    "change_date",
    last(col("date_parsed"), ignorenulls=True).over(date_fill_window)
).drop("date_parsed", "row_num")

# 2. Extract year from source_sheet for time-based analysis
df_transformed = df_transformed.withColumn(
    "change_year",
    substring(col("source_sheet"), 1, 4).cast("int")
)

# 3. Parse old_diff - handle special cases
# "no" appears in data (abbreviated "no change"), convert to null
df_transformed = df_transformed.withColumn(
    "old_difficulty",
    when(col("old_diff") == "no", None)
    .otherwise(col("old_diff").cast("int"))
)

# 4. Parse new_diff - handle multiple special cases
# Case 1: "no change" -> use old_diff value
# Case 2: "change" -> null (change occurred but new value not recorded)
# Case 3: "(resolved) X" -> extract X
# Case 4: numeric string -> convert to int
df_transformed = df_transformed.withColumn(
    "new_difficulty",
    when(col("new_diff") == "no change", col("old_difficulty"))
    .when(col("new_diff") == "change", None)
    .when(col("new_diff").contains("(resolved)"), 
          regexp_extract(col("new_diff"), r"\((resolved)\)\s*(\d+)", 2).cast("int"))
    .otherwise(col("new_diff").cast("int"))
)

# 5. Add flag for whether there was an actual change
df_transformed = df_transformed.withColumn(
    "is_actual_change",
    when(col("new_diff") == "no change", False)
    .otherwise(True)
)

# 6. Calculate difficulty change (new - old)
df_transformed = df_transformed.withColumn(
    "difficulty_change",
    when(col("new_difficulty").isNotNull() & col("old_difficulty").isNotNull(),
         col("new_difficulty") - col("old_difficulty"))
    .otherwise(None)
)

# 7. Calculate Reliability Weights
print("\nCalculating reliability weights...")

# 7a. Stability Score: 1 / (num_changes + 1)
# Count number of changes per song
song_change_counts = df_transformed.groupBy("song_name").agg(
    count_func("*").alias("num_changes")
)
df_transformed = df_transformed.join(song_change_counts, "song_name", "left")
df_transformed = df_transformed.withColumn(
    "stability_score",
    lit(1.0) / (col("num_changes") + 1)
)

# 7b. Recency Score: More recent = higher weight (DYNAMIC based on current year)
# Calculate years ago relative to current year
current_year = year_func(current_timestamp())
df_transformed = df_transformed.withColumn(
    "years_ago",
    current_year - col("change_year")
)
df_transformed = df_transformed.withColumn(
    "recency_score",
    when(col("years_ago") <= 0, 1.0)  # Current year or future
    .when(col("years_ago") == 1, 0.9)  # 1 year ago
    .when(col("years_ago") == 2, 0.8)  # 2 years ago
    .when(col("years_ago") == 3, 0.7)  # 3 years ago
    .otherwise(0.6)  # 4+ years ago
).drop("years_ago")

# 7c. Acceptance Score: Does proposed difficulty match current playlist?
# Load playlist to check if contested rating was accepted
df_playlist = spark.table("acubed.ffr.silver__playlist").select(
    col("name").alias("playlist_name"),
    col("difficulty").alias("current_difficulty")
)
df_transformed = df_transformed.join(
    df_playlist,
    df_transformed.song_name == df_playlist.playlist_name,
    "left"
)
df_transformed = df_transformed.withColumn(
    "acceptance_score",
    when(col("current_difficulty").isNull(), 0.7)  # Song not in playlist (neutral)
    .when(col("new_difficulty") == col("current_difficulty"), 1.0)  # Accepted
    .otherwise(0.5)  # Rejected/changed later
).drop("playlist_name", "current_difficulty")

# 7d. Magnitude Confidence: Smaller changes = higher confidence
df_transformed = df_transformed.withColumn(
    "magnitude_confidence",
    when(col("difficulty_change").isNull(), 0.5)  # Unknown change
    .when(col("difficulty_change").between(-2, 2), 1.0)  # Small adjustment
    .when(col("difficulty_change").between(-5, -3) | col("difficulty_change").between(3, 5), 0.8)  # Medium
    .when(col("difficulty_change") < -5, 0.6)  # Large decrease
    .when(col("difficulty_change") > 5, 0.6)  # Large increase
    .otherwise(0.5)
)

# 7e. Resolution Bonus: Was this marked as "(resolved)"?
df_transformed = df_transformed.withColumn(
    "resolution_bonus",
    when(col("new_diff").contains("(resolved)"), 0.2).otherwise(0.0)
)

# 7f. Combined Weight = stability × recency × acceptance × magnitude + resolution_bonus
df_transformed = df_transformed.withColumn(
    "reliability_weight",
    (col("stability_score") * col("recency_score") * col("acceptance_score") * col("magnitude_confidence")) + col("resolution_bonus")
)

print("✓ Reliability weights calculated")

# 8. Select and rename columns for silver table
df_silver = df_transformed.select(
    col("song_name"),
    col("old_difficulty"),
    col("new_difficulty"),
    col("difficulty_change"),
    col("is_actual_change"),
    col("change_date"),
    col("change_year"),
    col("source_sheet"),
    col("num_changes"),
    col("stability_score"),
    col("recency_score"),
    col("acceptance_score"),
    col("magnitude_confidence"),
    col("resolution_bonus"),
    col("reliability_weight"),
    col("ingestion_timestamp"),
    current_timestamp().alias("transformation_timestamp")
)

# 9. Apply data quality filters
print("\nApplying data quality filters...")
rows_before = df_silver.count()

# Filter out records with null new_difficulty (incomplete data from source)
df_silver = df_silver.filter(col("new_difficulty").isNotNull())
rows_after_null_filter = df_silver.count()
null_removed = rows_before - rows_after_null_filter

# Filter out records with new_difficulty = 0 (unrealistic minimum)
df_silver = df_silver.filter(col("new_difficulty") > 0)
rows_after_zero_filter = df_silver.count()
zero_removed = rows_after_null_filter - rows_after_zero_filter

# Filter out records with unrealistic difficulty values (> 120)
df_silver = df_silver.filter(col("new_difficulty") <= 120)
rows_after_max_filter = df_silver.count()
max_removed = rows_after_zero_filter - rows_after_max_filter

rows_final = df_silver.count()
total_removed = rows_before - rows_final

print(f"✓ Removed {null_removed} records with null new_difficulty")
print(f"✓ Removed {zero_removed} records with new_difficulty = 0")
print(f"✓ Removed {max_removed} records with new_difficulty > 120")
print(f"✓ Total filtered: {total_removed} records ({rows_before} → {rows_final})")

print("\n" + "="*60)
print("TRANSFORMATION COMPLETE")
print("="*60)
print(f"Rows transformed: {rows_final}")
print("\nWeight distribution:")
df_silver.select("reliability_weight").summary("mean", "min", "max", "stddev").show()
print("\nSample of transformed data:")
display(df_silver.limit(10))

In [0]:
# Validate the transformations
from pyspark.sql.functions import sum as spark_sum, count, when, col

print("="*60)
print("DATA QUALITY VALIDATION")
print("="*60)

# Check for nulls in key columns
print("\n--- Null Counts ---")
null_validation = df_silver.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in ["song_name", "old_difficulty", "new_difficulty", "change_date", "difficulty_change", "reliability_weight"]
])
null_validation.show(vertical=True)

# Summary statistics
print("\n--- Difficulty Change Statistics ---")
df_silver.filter(col("is_actual_change") == True).select(
    "difficulty_change"
).summary("count", "mean", "min", "max", "stddev").show()

# Distribution by year
print("\n--- Changes by Year ---")
df_silver.groupBy("change_year").count().orderBy("change_year").show()

# Reliability weight statistics
print("\n--- Reliability Weight Statistics ---")
df_silver.select("reliability_weight").summary("count", "mean", "min", "max", "stddev", "25%", "50%", "75%").show()

# Show highest and lowest weighted assessments
print("\n--- Highest Reliability Assessments (Top 10) ---")
df_silver.orderBy(col("reliability_weight").desc()).select(
    "song_name", "new_difficulty", "change_year", "num_changes", "reliability_weight"
).show(10, truncate=False)

print("\n--- Lowest Reliability Assessments (Bottom 10) ---")
df_silver.orderBy(col("reliability_weight").asc()).select(
    "song_name", "new_difficulty", "change_year", "num_changes", "reliability_weight"
).show(10, truncate=False)

# Show records with special values (resolved or unknown changes)
print("\n--- Special Cases ---")
print("Records where difficulty was resolved to specific value:")
df_bronze_check = spark.table("acubed.ffr.`bronze__contested-difficulties`")
df_bronze_check.filter(col("new_diff").contains("(resolved)")).select(
    "song_name", "old_diff", "new_diff", "source_sheet"
).show(truncate=False)

In [0]:
# Save transformed data to silver table
table_name = "acubed.ffr.`silver__contested-difficulties`"

# Capture count BEFORE write
rows_to_write = df_silver.count()

# Check if schema has changed (e.g., columns added/removed)
if spark.catalog.tableExists(table_name):
    existing_df = spark.table(table_name)
    existing_cols = set(existing_df.columns)
    new_cols = set(df_silver.columns)
    
    if existing_cols != new_cols:
        print(f"Schema has changed. Dropping and recreating table...")
        print(f"  Removed columns: {existing_cols - new_cols}")
        print(f"  Added columns: {new_cols - existing_cols}")
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")
        print(f"  Table dropped.")

if spark.catalog.tableExists(table_name):
    # Table exists with same schema - overwrite
    print(f"Table {table_name} exists. Overwriting with {rows_to_write} rows...")
    
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    
    print(f"✓ Successfully overwrote {table_name} with {rows_to_write} rows")
else:
    # Table doesn't exist - create it
    print(f"Creating new table {table_name} with {rows_to_write} rows...")
    
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    
    print(f"✓ Successfully created {table_name} with {rows_to_write} rows")

print("\n" + "="*60)
print("SILVER TABLE READY")
print("="*60)

In [0]:
from pyspark.sql.functions import sum as spark_sum, avg, stddev, count, max as max_func, min as min_func

print("="*60)
print("AGGREGATING RELIABILITY WEIGHTS")
print("="*60)

# 1. SONG-LEVEL AGGREGATION (for contested songs only)
print("\n--- Song-Level Weighted Aggregation ---")
df_song_weighted = df_silver.groupBy("song_name").agg(
    # Weighted average difficulty (weighted by reliability)
    (spark_sum(col("new_difficulty") * col("reliability_weight")) / 
     spark_sum(col("reliability_weight"))).alias("weighted_difficulty"),
    
    # Average reliability (represents confidence in this song's contested assessments)
    avg(col("reliability_weight")).alias("avg_reliability"),
    
    # Total reliability (sum of weights = overall confidence)
    spark_sum(col("reliability_weight")).alias("total_reliability"),
    
    # Number of contested assessments
    count("*").alias("num_assessments"),
    
    # Most recent assessment
    max_func(col("change_date")).alias("latest_change"),
    
    # Difficulty variance (uncertainty measure)
    stddev(col("new_difficulty")).alias("difficulty_stddev"),
    
    # Range of contested values
    min_func(col("new_difficulty")).alias("min_contested_difficulty"),
    max_func(col("new_difficulty")).alias("max_contested_difficulty")
)

print(f"Total contested songs: {df_song_weighted.count()}")
print("\nTop 10 most reliable contested songs:")
df_song_weighted.orderBy(col("total_reliability").desc()).show(10, truncate=False)

# 2. DIFFICULTY-LEVEL AGGREGATION (for all difficulty levels)
print("\n--- Difficulty-Level Reliability Stats ---")
df_difficulty_reliability = df_silver.groupBy("new_difficulty").agg(
    # Average reliability at this difficulty level
    avg(col("reliability_weight")).alias("avg_reliability"),
    
    # Standard deviation of reliability (variability in confidence)
    stddev(col("reliability_weight")).alias("stddev_reliability"),
    
    # Number of contested assessments at this difficulty
    count("*").alias("num_assessments"),
    
    # Total reliability (cumulative confidence)
    spark_sum(col("reliability_weight")).alias("total_reliability"),
    
    # Average recency score (how recent are assessments at this level?)
    avg(col("recency_score")).alias("avg_recency"),
    
    # Acceptance rate (what % matched current playlist?)
    avg(col("acceptance_score")).alias("avg_acceptance")
).orderBy("new_difficulty")

print(f"Difficulty levels with contested data: {df_difficulty_reliability.count()}")
print("\nReliability by difficulty level:")
df_difficulty_reliability.show(100, truncate=False)

print("\n" + "="*60)
print("HOW TO USE THESE AGGREGATIONS")
print("="*60)
print("""
For PREDICTION MODELS:

1. SONG-LEVEL (df_song_weighted):
   - Use when song EXISTS in contested table
   - weighted_difficulty = best estimate from contested assessments
   - total_reliability = confidence in this estimate (use as sample weight)
   - difficulty_stddev = uncertainty (for confidence intervals)

2. DIFFICULTY-LEVEL (df_difficulty_reliability):
   - Use as PRIOR for all songs at that difficulty
   - avg_reliability = baseline confidence for predictions at this difficulty
   - Low avg_reliability = model should be less confident in predictions
   - num_assessments = how much scrutiny this difficulty level has received

3. COMBINED APPROACH:
   For each song in training/prediction:
   - If song in df_song_weighted: Use weighted_difficulty with total_reliability weight
   - If song NOT contested: Use playlist difficulty with difficulty-level avg_reliability
   - This gives you both individual corrections AND population-level confidence

Example: Training sample weights:
   weight = song_total_reliability (if contested) OR difficulty_avg_reliability (if not)
""")

In [0]:
# Save difficulty-level reliability scores to a dedicated table
# Generate complete range 1-120 with defaults for missing difficulties
from pyspark.sql.functions import coalesce, lit

table_name = "acubed.ffr.`silver__reliability-scores`"

print("="*60)
print("SAVING RELIABILITY SCORES TABLE")
print("="*60)

# Create a complete range of difficulties (1-120)
print("\nGenerating complete difficulty range (1-120)...")
full_difficulty_range = spark.range(1, 121).select(col("id").cast("int").alias("new_difficulty"))

# Left join contested reliability data to the full range
df_complete_reliability = full_difficulty_range.join(
    df_difficulty_reliability,
    "new_difficulty",
    "left"
)

# Fill missing values with conservative defaults
DEFAULT_RELIABILITY = 0.20  # Conservative baseline for uncontested difficulties

df_complete_reliability = df_complete_reliability.select(
    col("new_difficulty"),
    coalesce(col("avg_reliability"), lit(DEFAULT_RELIABILITY)).alias("avg_reliability"),
    col("stddev_reliability"),  # Keep as null for uncontested (no variability data)
    coalesce(col("num_assessments"), lit(0)).alias("num_assessments"),
    coalesce(col("total_reliability"), lit(DEFAULT_RELIABILITY)).alias("total_reliability"),
    coalesce(col("avg_recency"), lit(0.6)).alias("avg_recency"),  # Default to 4+ years ago score
    coalesce(col("avg_acceptance"), lit(0.7)).alias("avg_acceptance")  # Neutral default
).orderBy("new_difficulty")

rows_to_write = df_complete_reliability.count()
contested_difficulties = df_complete_reliability.filter(col("num_assessments") > 0).count()
default_difficulties = df_complete_reliability.filter(col("num_assessments") == 0).count()

print(f"\n✓ Complete range generated:")
print(f"  - Total difficulties: {rows_to_write}")
print(f"  - With contested data: {contested_difficulties}")
print(f"  - Using defaults: {default_difficulties}")
print(f"  - Default reliability: {DEFAULT_RELIABILITY}")

if spark.catalog.tableExists(table_name):
    print(f"\nTable {table_name} exists. Overwriting with {rows_to_write} rows...")
    
    df_complete_reliability.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    
    print(f"✓ Successfully overwrote {table_name} with {rows_to_write} rows")
else:
    print(f"\nCreating new table {table_name} with {rows_to_write} rows...")
    
    df_complete_reliability.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    
    print(f"✓ Successfully created {table_name} with {rows_to_write} rows")

print("\n" + "="*60)
print("RELIABILITY SCORES TABLE READY")
print("="*60)
print(f"""
Table: {table_name}
Rows: {rows_to_write} difficulty levels (complete range 1-120)

Default Values for Uncontested Difficulties:
  - avg_reliability: {DEFAULT_RELIABILITY}
  - num_assessments: 0
  - total_reliability: {DEFAULT_RELIABILITY}
  - avg_recency: 0.6
  - avg_acceptance: 0.7
  - stddev_reliability: null (no variability data)

Usage:
  Join this table to your playlist/training data on difficulty level.
  ALL difficulty values 1-120 are guaranteed to have a reliability score.
  
Example:
  df_training.join(
      spark.table('{table_name}'),
      df_training.difficulty == col('new_difficulty'),
      'left'
  )
""")

In [0]:
# Apply ordinal smoothing to respect the sequential nature of difficulty values
from pyspark.sql.window import Window
from pyspark.sql.functions import avg as avg_func, col, lit, coalesce

print("="*60)
print("APPLYING ORDINAL SMOOTHING TO RELIABILITY SCORES")
print("="*60)

# Load the complete reliability scores (1-120)
df_reliability_raw = spark.table("acubed.ffr.`silver__reliability-scores`")

print("\nBefore smoothing:")
print(f"  Total difficulties: {df_reliability_raw.count()}")
print(f"  avg_reliability range: {df_reliability_raw.agg({'avg_reliability': 'min'}).collect()[0][0]:.3f} - {df_reliability_raw.agg({'avg_reliability': 'max'}).collect()[0][0]:.3f}")

# Define window size for smoothing (use neighboring difficulties)
WINDOW_SIZE = 5  # Will average across ±2 difficulties (5-point window)

# Create a rolling window based on difficulty value (ordinal)
window_spec = Window.orderBy("new_difficulty").rowsBetween(-WINDOW_SIZE // 2, WINDOW_SIZE // 2)

# Apply smoothing to avg_reliability
df_smoothed = df_reliability_raw.withColumn(
    "avg_reliability_raw",
    col("avg_reliability")  # Keep original for comparison
).withColumn(
    "avg_reliability_smoothed",
    avg_func(col("avg_reliability")).over(window_spec)
)

# Replace avg_reliability with smoothed version
df_smoothed = df_smoothed.withColumn(
    "avg_reliability",
    col("avg_reliability_smoothed")
).drop("avg_reliability_smoothed")

print(f"\n✓ Applied {WINDOW_SIZE}-point rolling window smoothing")
print(f"  Window: ±{WINDOW_SIZE // 2} difficulties")
print(f"\nAfter smoothing:")
print(f"  avg_reliability range: {df_smoothed.agg({'avg_reliability': 'min'}).collect()[0][0]:.3f} - {df_smoothed.agg({'avg_reliability': 'max'}).collect()[0][0]:.3f}")

# Show before/after comparison for sample difficulties
print("\n--- Before/After Comparison (Sample) ---")
df_smoothed.select(
    "new_difficulty",
    col("avg_reliability_raw").alias("before_smooth"),
    col("avg_reliability").alias("after_smooth"),
    (col("avg_reliability") - col("avg_reliability_raw")).alias("change"),
    "num_assessments"
).filter(
    (col("new_difficulty") % 10 == 0) |  # Every 10th difficulty
    (col("num_assessments") == 0)  # Show some uncontested
).orderBy("new_difficulty").show(30, truncate=False)

# Update the table with smoothed values
table_name = "acubed.ffr.`silver__reliability-scores`"

print(f"\nUpdating {table_name} with smoothed reliability scores...")

# Keep only the final columns (drop raw comparison column)
df_final = df_smoothed.select(
    "new_difficulty",
    "avg_reliability",
    "stddev_reliability",
    "num_assessments",
    "total_reliability",
    "avg_recency",
    "avg_acceptance"
).orderBy("new_difficulty")

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"✓ Successfully updated {table_name} with smoothed scores")

print("\n" + "="*60)
print("SMOOTHING BENEFITS")
print("="*60)
print(f"""
Why Ordinal Smoothing?
  ✓ Respects the sequential nature of difficulty values
  ✓ Reduces noise from single assessments at isolated difficulties
  ✓ Provides more stable and meaningful defaults for uncontested levels
  ✓ Creates smooth transitions across the difficulty spectrum
  ✓ Leverages information from neighboring difficulties

Window Size: {WINDOW_SIZE} difficulties (±{WINDOW_SIZE // 2})
  - Smaller window: More responsive to local patterns, more noise
  - Larger window: Smoother curve, may over-smooth
  - {WINDOW_SIZE} is a balanced choice for 120-point scale

Result:
  - Uncontested difficulties now interpolate from nearby contested levels
  - Sharp jumps in reliability are smoothed out
  - Model training will benefit from more stable sample weights
""")

# Visualize the smoothing effect
print("\n--- Reliability Score Distribution (After Smoothing) ---")
df_final.select("avg_reliability").summary("count", "mean", "min", "max", "stddev", "25%", "50%", "75%").show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np

print("="*60)
print("VISUALIZING SMOOTHING EFFECT")
print("="*60)

# Collect data from the smoothed dataframe (which still has avg_reliability_raw)
df_viz = df_smoothed.select(
    "new_difficulty",
    "avg_reliability_raw",
    "avg_reliability",
    "num_assessments"
).orderBy("new_difficulty").toPandas()

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Subplot 1: Raw vs Smoothed Reliability Scores
ax1.plot(df_viz['new_difficulty'], df_viz['avg_reliability_raw'], 
         'o-', alpha=0.4, color='#1f77b4', label='Raw (Before Smoothing)', markersize=3)
ax1.plot(df_viz['new_difficulty'], df_viz['avg_reliability'], 
         'o-', alpha=0.8, color='#ff7f0e', label='Smoothed (After)', markersize=3, linewidth=2)

# Highlight uncontested difficulties (num_assessments == 0)
uncontested = df_viz[df_viz['num_assessments'] == 0]
ax1.scatter(uncontested['new_difficulty'], uncontested['avg_reliability_raw'], 
           color='red', s=50, alpha=0.5, marker='x', label='Uncontested (used default 0.20)', zorder=5)

ax1.set_xlabel('Difficulty Level', fontsize=12, fontweight='bold')
ax1.set_ylabel('avg_reliability (Sample Weight)', fontsize=12, fontweight='bold')
ax1.set_title('Reliability Scores: Before vs After Ordinal Smoothing (5-Point Window)', 
              fontsize=14, fontweight='bold', pad=20)
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_xlim(0, 121)
ax1.set_ylim(0.15, 0.45)

# Add annotations for key insights
ax1.text(0.98, 0.05, 
         'Smoothing Effect:\n• Eliminates sharp jumps\n• Interpolates uncontested values\n• Preserves overall trend',
         transform=ax1.transAxes, fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

# Subplot 2: Change from Smoothing (Difference)
df_viz['change'] = df_viz['avg_reliability'] - df_viz['avg_reliability_raw']

colors = ['green' if x > 0 else 'red' if x < 0 else 'gray' for x in df_viz['change']]
ax2.bar(df_viz['new_difficulty'], df_viz['change'], color=colors, alpha=0.6, width=1.0)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)

ax2.set_xlabel('Difficulty Level', fontsize=12, fontweight='bold')
ax2.set_ylabel('Change in avg_reliability', fontsize=12, fontweight='bold')
ax2.set_title('Impact of Smoothing: Change in Reliability Score by Difficulty', 
              fontsize=14, fontweight='bold', pad=20)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')
ax2.set_xlim(0, 121)

# Add statistics
max_increase = df_viz['change'].max()
max_decrease = df_viz['change'].min()
mean_abs_change = df_viz['change'].abs().mean()

ax2.text(0.02, 0.95, 
         f'Largest increase: +{max_increase:.3f}\nLargest decrease: {max_decrease:.3f}\nMean abs. change: {mean_abs_change:.3f}',
         transform=ax2.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete!")
print("\nKey Observations:")
print(f"  • Uncontested difficulties (red X): {len(uncontested)} levels")
print(f"  • Contested difficulties: {len(df_viz) - len(uncontested)} levels")
print(f"  • Average change magnitude: {mean_abs_change:.4f}")
print(f"  • Largest improvement (uncontested): +{max_increase:.4f}")
print(f"  • Largest reduction (outlier): {max_decrease:.4f}")
print("\nSmoothing Benefits Visible:")
print("  ✓ Orange line (smoothed) is much smoother than blue (raw)")
print("  ✓ Uncontested difficulties (red X) now have meaningful values from neighbors")
print("  ✓ Sharp spikes in raw data are dampened")
print("  ✓ Overall trend preserved while reducing noise")

In [0]:
# Example: Join reliability scores to playlist with default weight for missing difficulties
from pyspark.sql.functions import coalesce, lit

print("="*60)
print("JOINING RELIABILITY SCORES TO PLAYLIST")
print("="*60)

# Load playlist
df_playlist = spark.table("acubed.ffr.silver__playlist")

# Load reliability scores
df_reliability = spark.table("acubed.ffr.`silver__reliability-scores`")

# Join with LEFT join (keeps all songs, even if no reliability score)
df_with_reliability = df_playlist.join(
    df_reliability,
    df_playlist.difficulty == df_reliability.new_difficulty,
    "left"
)

# Add default weight for missing difficulties
DEFAULT_WEIGHT = 0.20  # Conservative baseline for uncontested difficulties (lower than contested range)

df_with_reliability = df_with_reliability.withColumn(
    "sample_weight",
    coalesce(col("avg_reliability"), lit(DEFAULT_WEIGHT))
)

print(f"\nTotal songs in playlist: {df_playlist.count()}")
print(f"Songs with contested reliability scores: {df_with_reliability.filter(col('avg_reliability').isNotNull()).count()}")
print(f"Songs using default weight ({DEFAULT_WEIGHT}): {df_with_reliability.filter(col('avg_reliability').isNull()).count()}")

# Show distribution of weights
print("\n--- Sample Weight Distribution ---")
df_with_reliability.select("sample_weight").summary("count", "mean", "min", "max", "stddev", "25%", "50%", "75%").show()

# Show examples of songs with and without reliability scores
print("\n--- Sample: Songs WITH contested reliability ---")
df_with_reliability.filter(col("avg_reliability").isNotNull()).select(
    "name", "difficulty", "avg_reliability", "num_assessments", "sample_weight"
).orderBy("difficulty").show(10, truncate=False)

print("\n--- Sample: Songs WITHOUT contested reliability (using default) ---")
df_with_reliability.filter(col("avg_reliability").isNull()).select(
    "name", "difficulty", "sample_weight"
).orderBy("difficulty").show(10, truncate=False)

print("\n" + "="*60)
print("WEIGHT STRATEGY SUMMARY")
print("="*60)
print(f"""
Default Weight: {DEFAULT_WEIGHT}
Rationale: 
  - CONSERVATIVE approach for uncontested difficulties
  - Lower difficulty levels (0-50) are not well-established and have variable reliability
  - Prevents model from over-trusting songs that haven't been scrutinized
  - Still allows learning but with appropriate skepticism
  - Lower than most contested scores (avg_reliability range: 0.12-0.40, mean: 0.278)

Alternative Strategies:
  1. Use mean of all avg_reliability: {df_reliability.agg({'avg_reliability': 'mean'}).collect()[0][0]:.3f} (less conservative)
  2. Use 0.15 (very conservative, for highly uncertain domains)
  3. Use 0.25 (balanced, closer to population mean)
  4. Use difficulty-based interpolation (complex, may not be worth it)
""")

In [0]:
%sql
select * from acubed.ffr.`silver__reliability-scores`